In [0]:
# ============================================================
# Bronze — Source 03: MongoDB Atlas
#
# Incremental load using watermark pattern.
# Only processes S3 files newer than last successful run.
#
# Source: s3://ecommerce-lakehouse-467091806172-raw-01/source=03_mongodb/
# Target: bronze.mongodb catalog in Unity Catalog
#
# Tables:
#   - bronze.mongodb.products
# ============================================================

import sys
sys.path.append("/Workspace/Users/sutharripal26@gmail.com/ecommerce-lakehouse/pipelines/bronze/shared")
from bronze_utils import get_watermark, update_watermark
from pyspark.sql.functions import col, lit, max as spark_max

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

RAW_BUCKET = "s3://ecommerce-lakehouse-467091806172-raw-01"
SOURCE = "03_mongodb"

In [0]:
# Get watermark — epoch on first run
watermark = get_watermark(spark, SOURCE)
print(f"Loading files modified after: {watermark}")

path = f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=22/"

df = spark.read.format("parquet") \
    .load(path) \
    .filter(col("_metadata.file_modification_time") > lit(watermark))

row_count = df.count()

if row_count == 0:
    print(f"[{SOURCE}] No new data — skipping")
else:
    spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.mongodb")
    
    df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("bronze.mongodb.products")
    
    latest_ts = spark.read.format("parquet") \
        .load(path) \
        .select(spark_max("_metadata.file_modification_time")).collect()[0][0]
    
    update_watermark(spark, SOURCE, latest_ts, row_count)
    print(f"✅ bronze.mongodb.products: {row_count} rows loaded")

In [0]:
count = spark.sql("SELECT COUNT(*) as cnt FROM bronze.mongodb.products").collect()[0]['cnt']
print(f"bronze.mongodb.products: {count} rows")

spark.sql("SELECT * FROM bronze.pipeline.watermarks WHERE source = '03_mongodb'").show()